In [1]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
#from sklearn import datasets

In [2]:
df = sns.load_dataset("titanic")
df.head()
# import Titanic dataset
# data = datasets.fetch_titanic



,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [3]:
from sklearn.datasets import fetch_openml

titanic = fetch_openml("titanic", version=1, as_frame=True)
df1 = titanic.frame
df1.head()


,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


In [ ]:
# Basic inspection
print(df.head())
print(df.shape)
print(df.info())


In [3]:
missing = pd.DataFrame({
    "Missing": df.isnull().sum(),
    "Percentage": (df.isnull().sum() / len(df) * 100).round(2)
})

print(missing.sort_values("Percentage", ascending=False))

             Missing  Percentage
deck             688       77.22
age              177       19.87
embarked           2        0.22
embark_town        2        0.22
sex                0        0.00
pclass             0        0.00
survived           0        0.00
fare               0        0.00
parch              0        0.00
sibsp              0        0.00
class              0        0.00
adult_male         0        0.00
who                0        0.00
alive              0        0.00
alone              0        0.00


In [6]:
df["age"].describe()

count    714.000000
mean      29.699118
std       14.526497
min        0.420000
25%       20.125000
50%       28.000000
75%       38.000000
max       80.000000
Name: age, dtype: float64

In [7]:
df["age"].fillna(df["age"].median())

0      22.0
1      38.0
2      26.0
3      35.0
4      35.0
       ... 
886    27.0
887    19.0
888    28.0
889    26.0
890    32.0
Name: age, Length: 891, dtype: float64

In [8]:
df["age"] = df["age"].fillna(
    df.groupby(["sex", "pclass"])["age"].transform("median")
)

In [9]:
df["embarked"].value_counts(dropna=False)

embarked
S      644
C      168
Q       77
NaN      2
Name: count, dtype: int64

In [10]:
df["embarked"] = df["embarked"].fillna(
    df["embarked"].mode()[0]
)

In [11]:
df["deck"].value_counts(dropna=False)

deck
NaN    688
C       59
B       47
D       33
E       32
A       15
F       13
G        4
Name: count, dtype: int64

In [13]:
df["deck_known"] = df["deck"].notna()
df[["deck", "deck_known"]].head()

,deck,deck_known
0,NaN,False
1,C,True
2,NaN,False
3,C,True
4,NaN,False


In [14]:
df = df.drop(columns=["deck"])

## Decision
Preserve whether deck information was available, then remove deck.
### Justification
deck contains too many missing values for reliable imputation. However, the availability of cabin information itself may contain useful information. Therefore, a binary deck_known variable is created before removing the original variable.

### This is a good example of:
Preserve useful information before removing a variable.

In [15]:
df[["embarked", "embark_town"]].head(10)

,embarked,embark_town
0,S,Southampton
1,C,Cherbourg
2,S,Southampton
3,S,Southampton
4,S,Southampton
5,Q,Queenstown
6,S,Southampton
7,S,Southampton
8,S,Southampton
9,C,Cherbourg


In [16]:
df = df.drop(columns=["embark_town"])

### Justification
embarked and embark_town represent the same information. Retaining both introduces unnecessary redundancy.

In [17]:
df.dtypes

survived         int64
pclass           int64
sex                str
age            float64
sibsp            int64
parch            int64
fare           float64
embarked           str
class         category
who                str
adult_male        bool
alive              str
alone             bool
deck_known        bool
dtype: object

In [18]:
df["pclass"] = df["pclass"].astype("category")

In [19]:
df["sex"] = df["sex"].astype("category")
df["embarked"] = df["embarked"].astype("category")

In [20]:
df["survived"] = df["survived"].astype("category")

In [21]:
df.dtypes

survived      category
pclass        category
sex           category
age            float64
sibsp            int64
parch            int64
fare           float64
embarked      category
class         category
who                str
adult_male        bool
alive              str
alone             bool
deck_known        bool
dtype: object

In [22]:
df.duplicated().sum()

np.int64(113)

In [23]:
df[df.duplicated()].head(10)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,alive,alone,deck_known
47,1,3,female,21.5,0,0,7.7500,Q,Third,woman,False,yes,True,False
64,0,1,male,40.0,0,0,27.7208,C,First,man,True,no,True,False
76,0,3,male,25.0,0,0,7.8958,S,Third,man,True,no,True,False
77,0,3,male,25.0,0,0,8.0500,S,Third,man,True,no,True,False
87,0,3,male,25.0,0,0,8.0500,S,Third,man,True,no,True,False
95,0,3,male,25.0,0,0,8.0500,S,Third,man,True,no,True,False
101,0,3,male,25.0,0,0,7.8958,S,Third,man,True,no,True,False
121,0,3,male,25.0,0,0,8.0500,S,Third,man,True,no,True,False
133,1,2,female,29.0,1,0,26.0000,S,Second,woman,False,yes,False,False
173,0,3,male,21.0,0,0,7.9250,S,Third,man,True,no,True,False


## Decision
Check duplicates, but do not automatically remove them.
### Justification
Duplicate-looking records cannot confidently be classified as erroneous because the dataset lacks a unique passenger identifier. Removing them could accidentally remove legitimate passengers.


### Duplicate rows ≠ necessarily duplicate entities.

In [24]:
df[["age", "fare", "sibsp", "parch"]].describe()

,age,fare,sibsp,parch
count,891.000000,891.000000,891.000000,891.000000
mean,29.112424,32.204208,0.523008,0.381594
std,13.304424,49.693429,1.102743,0.806057
min,0.420000,0.000000,0.000000,0.000000
25%,21.500000,7.910400,0.000000,0.000000
50%,26.000000,14.454200,0.000000,0.000000
75%,36.000000,31.000000,1.000000,0.000000
max,80.000000,512.329200,8.000000,6.000000


In [25]:
df[df["age"] < 0]

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,alive,alone,deck_known


In [26]:
df[df["fare"] < 0]

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,alive,alone,deck_known


In [27]:
df["pclass"].value_counts()

pclass
3    491
1    216
2    184
Name: count, dtype: int64